# SMOG3 Quickstart

This notebook demonstrates a lightweight SMOG3 workflow: check the install, download or locate a small PDB, generate all-atom and C-alpha models, and optionally compare with official SMOG2 Docker.

In [ ]:
import shutil
import subprocess
from pathlib import Path

import smog3

print("SMOG3", smog3.__version__)
print("smog3 command:", shutil.which("smog3"))
print("java command:", shutil.which("java"))

## Prepare 2CI2

The cell downloads 2CI2 from RCSB when network access is available.  In a source checkout, it can fall back to the local SMOG-CHECK copy.

In [ ]:
pdb = Path("2CI2.pdb")
if not pdb.exists():
    try:
        import urllib.request
        urllib.request.urlretrieve("https://files.rcsb.org/download/2CI2.pdb", pdb)
    except Exception:
        fallback = Path("../SMOG-CHECK/share/PDB.files/2ci2_v2.pdb")
        if fallback.exists():
            pdb.write_bytes(fallback.read_bytes())
        else:
            raise
print(pdb, pdb.stat().st_size, "bytes")

## Run All-Atom SMOG3

In [ ]:
aa_cmd = [
    "smog3", "-i", str(pdb), "-AA",
    "-o", "notebook_aa.top",
    "-g", "notebook_aa.gro",
    "-n", "notebook_aa.ndx",
    "-s", "notebook_aa.contacts",
]
subprocess.run(aa_cmd, check=True)
for name in ["notebook_aa.top", "notebook_aa.gro", "notebook_aa.ndx", "notebook_aa.contacts"]:
    path = Path(name)
    print(name, path.stat().st_size, "bytes")

## Run C-Alpha SMOG3

In [ ]:
ca_cmd = [
    "smog3", "-i", str(pdb), "-CA",
    "-o", "notebook_ca.top",
    "-g", "notebook_ca.gro",
    "-n", "notebook_ca.ndx",
    "-s", "notebook_ca.contacts",
]
subprocess.run(ca_cmd, check=True)
for name in ["notebook_ca.top", "notebook_ca.gro", "notebook_ca.ndx", "notebook_ca.contacts"]:
    path = Path(name)
    print(name, path.stat().st_size, "bytes")

## Optional OpenSMOG XML

In [ ]:
opensmog_cmd = [
    "smog3", "-i", str(pdb), "-AA", "-OpenSMOG", "-OpenSMOGxml", "notebook.xml",
    "-o", "notebook_os.top",
    "-g", "notebook_os.gro",
    "-n", "notebook_os.ndx",
    "-s", "notebook_os.contacts",
]
subprocess.run(opensmog_cmd, check=True)
print(Path("notebook.xml").read_text().splitlines()[0])

## Optional Docker Comparison

From a source checkout with Docker available, run:

```bash
bash scripts/validate_real_pdb_panel.sh --local-only --cases protein_ci2
```